# Dual-Signal SBFL Pipeline for NL-to-FOL Prolog KB Repair on FOLIO

This notebook demonstrates the **dual-signal spectrum-based fault localization (SBFL)** pipeline for repairing LLM-extracted Prolog knowledge bases on the FOLIO natural-language reasoning benchmark.

## What this artifact does

1. **Extracts** a Prolog KB from natural-language premises via LLM
2. **Generates** oracle yes/no questions to probe KB correctness
3. **Converts** oracle questions to Prolog goals (batched to 1 LLM call)
4. **Runs** each goal through a meta-interpreter tracking Ochiai suspiciousness scores (direct-fault predicates) and sub-goal harvest scores (missing predicates)
5. **Repairs** top-K suspicious items via LLM
6. **Evaluates** the conclusion using the repaired KB

**Methods compared:** `dual_sbfl` (ours), `oneshot`, `cot`, `selfrefine`, `flat_sbfl` (ablation)

**Dataset:** FOLIO validation split (203 examples) — natural-language reasoning requiring FOL-like inference

> This notebook loads **pre-computed results** from `mini_demo_data.json` to illustrate the pipeline outputs and compare methods, without requiring SWI-Prolog or API keys at demo time.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru, tqdm, requests — NOT pre-installed on Colab, always install
_pip('loguru==0.7.3', 'tqdm==4.68.1', 'requests==2.34.2')

# Core packages — pre-installed on Colab, install locally only to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import gc
import json
import math
import os
import re
import tempfile
import time
from collections import Counter
from pathlib import Path
from typing import Any

import requests
from loguru import logger
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Loading

We load pre-computed results from `mini_demo_data.json`. The data contains 3 FOLIO examples with predictions from all 5 methods: `dual_sbfl`, `oneshot`, `cot`, `selfrefine`, and `flat_sbfl`.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d8d250-dual-signal-spectrum-based-fault-localiz/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data: {data['metadata']['description']}")
print(f"Model: {data['metadata']['model']}")
print(f"Examples: {data['metadata']['summary']['n_examples']}")

## Configuration

All tunable pipeline parameters. The demo uses the same values as the original run (3-example mini set).
To reproduce the full 203-example run, set `MAX_EXAMPLES = 203`.

In [ ]:
MODEL = "meta-llama/llama-3.1-8b-instruct"
FALLBACK_MODEL = "google/gemma-2-9b-it"
MAX_BUDGET_USD = 9.5
MAX_EXAMPLES = 3          # minimum demo; original full run: 203
N_ORACLE_QUERIES = 5      # yes/no oracle questions per example; original: 5
N_REPAIR_ROUNDS = 1       # SBFL repair iterations; original: 1
K_REPAIR_TARGETS = 3      # top-K suspicious predicates to repair; original: 3
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

## LLM Cost Tracker and API Helper

The pipeline makes multiple LLM calls per example. `COST_TRACKER` accumulates usage; `llm_call` wraps the OpenRouter API with retry logic and per-call cost accounting using approximate token prices.

In [ ]:
COST_TRACKER: dict[str, float] = {"total": 0.0, "calls": 0}

# Approximate pricing per million tokens (OpenRouter, as of 2025)
MODEL_PRICES: dict[str, tuple[float, float]] = {
    "meta-llama/llama-3.1-8b-instruct": (0.055, 0.055),
    "google/gemma-2-9b-it": (0.07, 0.07),
    "anthropic/claude-3-haiku": (0.25, 1.25),
}


class BudgetExceededError(RuntimeError):
    pass


def llm_call(
    messages: list[dict],
    model: str = MODEL,
    max_tokens: int = 512,
    temperature: float = 0.0,
    retries: int = 3,
) -> tuple[str, float]:
    """Call OpenRouter API. Returns (text, cost_usd)."""
    if COST_TRACKER["total"] >= MAX_BUDGET_USD:
        raise BudgetExceededError(f"Budget ${MAX_BUDGET_USD} exceeded")

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    last_err: Exception | None = None
    for attempt in range(retries):
        try:
            resp = requests.post(
                f"{OPENROUTER_BASE}/chat/completions",
                headers=headers,
                json=payload,
                timeout=120,
            )
            if resp.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue
            resp.raise_for_status()
            data = resp.json()
            text = data["choices"][0]["message"]["content"] or ""
            usage = data.get("usage", {})
            in_tok = usage.get("prompt_tokens", 0)
            out_tok = usage.get("completion_tokens", 0)
            in_price, out_price = MODEL_PRICES.get(model, (0.1, 0.1))
            cost = (in_tok * in_price + out_tok * out_price) / 1_000_000
            COST_TRACKER["total"] += cost
            COST_TRACKER["calls"] += 1
            return text, cost
        except BudgetExceededError:
            raise
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))

    return "", 0.0

## Prolog Meta-Interpreter

The SBFL pipeline runs oracle Prolog goals through a custom meta-interpreter that logs coverage (succeeded/failed/depth_exceeded) for each predicate. This is how the Ochiai suspiciousness scores are computed: failing test cases that exercise a predicate more than passing ones indicate it is buggy.

In [ ]:
META_INTERPRETER_PROLOG = """\
:- dynamic coverage_log/3.
:- dynamic failed_subgoal/1.

reset_coverage :-
    retractall(coverage_log(_, _, _)),
    retractall(failed_subgoal(_)).

% Entry point with depth limit to prevent stack overflow from cyclic KBs
solve_safe(Goal, D) :-
    catch(
        call_with_depth_limit(solve(Goal, D), 50, _),
        _,
        fail
    ).

solve(true, _) :- !.
solve(fail, _) :- !, fail.
solve((A, B), D) :- !, (D > 0 -> D1 is D-1 ; D1 = 0), solve(A, D1), solve(B, D1).
solve((A ; B), D) :- !, (solve(A, D) ; solve(B, D)).
solve(\\+(A), D) :- !, (solve(A, D) -> fail ; true).
solve(Goal, D) :-
    D > 0,
    functor(Goal, Name, Arity),
    (catch(clause(Goal, Body), _, fail) ->
        D1 is D - 1,
        (solve(Body, D1) ->
            assertz(coverage_log(Name, Arity, succeeded))
        ;
            assertz(coverage_log(Name, Arity, subgoal_failed)),
            assertz(failed_subgoal(Goal)),
            fail
        )
    ;
        assertz(coverage_log(Name, Arity, unify_failed)),
        assertz(failed_subgoal(Goal)),
        fail
    ).
solve(Goal, 0) :-
    functor(Goal, Name, Arity),
    assertz(coverage_log(Name, Arity, depth_exceeded)),
    fail.
"""

print("Meta-interpreter Prolog code defined (", len(META_INTERPRETER_PROLOG.splitlines()), "lines)")

## Clause Sanitization

LLM-generated Prolog often contains syntax errors. `sanitize_clause` filters out: negation-as-facts, conjunction-in-head, bare `not X`, CamelCase predicate heads, markdown/comment artifacts, and other invalid patterns.

In [ ]:
_PREDICATE_SIGNATURE_RE = re.compile(r"^[a-z][a-zA-Z0-9_]*/\d+$")
_PROLOG_DANGEROUS_RE = re.compile(r"\b(assert|retract|abolish|consult|halt|write|read|open|close|nl)\b")


def sanitize_clause(clause: str) -> str | None:
    """Return a safe, valid-looking Prolog clause or None to discard."""
    c = clause.strip().rstrip(".")
    if not c:
        return None
    # Discard predicate-signature lines like  foo/1
    if _PREDICATE_SIGNATURE_RE.match(c):
        return None
    # Discard meta/dangerous predicates
    if _PROLOG_DANGEROUS_RE.search(c):
        return None
    # Remove any comment suffix  (% ...)
    c = re.sub(r"\s*%.*$", "", c).strip()
    # Replace 'not(X)' → '\+(X)' and 'not(X,...)' patterns
    c = re.sub(r"\bnot\s*\(", r"\\+(", c)
    # Discard any clause with remaining bare 'not ' (LLM used invalid syntax)
    if re.search(r"\bnot\s+[a-zA-Z_]", c):
        return None
    # Discard facts that are negations - \+(foo) can't be asserted as a fact
    if c.lstrip().startswith("\\+"):
        return None
    # Discard clauses with conjunctions in the head (top-level comma before :-)
    head_part = c.split(":-")[0].strip()
    depth = 0
    for ch in head_part:
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth -= 1
        elif ch == "," and depth == 0:
            return None  # conjunction in head is invalid Prolog
    # Discard lines that look like comments slipped through
    if c.startswith("/*") or c.startswith("*/"):
        return None
    # Discard numbered-list items or markdown code that slipped through
    if re.match(r"^\d+\.", c) or "`" in c or c.startswith("-") or c.startswith("*"):
        return None
    # Discard lines starting with uppercase (not valid Prolog clause heads)
    if re.match(r"^[A-Z]", c.split(":-")[0].strip()):
        return None
    c = c.strip()
    if not c:
        return None
    return c + "."


def extract_predicate_names_from_clauses(clauses: list[str]) -> list[tuple[str, int]]:
    preds: set[tuple[str, int]] = set()
    for clause in clauses:
        head = clause.split(":-")[0].strip().rstrip(".")
        m = re.match(r"([a-z][a-zA-Z0-9_]*)\((.*)\)", head, re.DOTALL)
        if m:
            name = m.group(1)
            args_str = m.group(2).strip()
            # count top-level commas
            depth = 0
            arity = 1 if args_str else 0
            for ch in args_str:
                if ch in "([":
                    depth += 1
                elif ch in ")]":
                    depth -= 1
                elif ch == "," and depth == 0:
                    arity += 1
            preds.add((name, arity))
        else:
            plain = re.match(r"([a-z][a-zA-Z0-9_]*)$", head)
            if plain:
                preds.add((plain.group(1), 0))
    return list(preds)


# Quick demo of the sanitizer
test_clauses = [
    "is_happy(john).",
    "not(is_sad(john)).",          # negation-as-fact — invalid
    "Foo(bar).",                    # CamelCase head — invalid
    "parent_of(X, Y) :- foo/1.",   # predicate sig leaked in body — dangerous
    "student(alice) :- attends_school(alice), is_enrolled(alice).",
]
for c in test_clauses:
    result = sanitize_clause(c)
    print(f"  {c!r:60s} → {result!r}")

## Ochiai SBFL Score

The Ochiai formula computes suspiciousness for each predicate: `ef / sqrt((ef+nf)*(ef+ep))` where `ef` = failing tests that exercised the predicate, `ep` = passing tests that exercised it, `nf` = failing tests that didn't.

The **dual-signal** twist: beyond Ochiai (direct-fault), the pipeline also harvests sub-goals that failed unification — these represent *missing* predicates not yet in the KB.

In [ ]:
def compute_ochiai_scores(
    coverage_matrix: dict[str, list[str | None]],
    pass_fail: list[bool],
) -> dict[str, float]:
    scores: dict[str, float] = {}
    n_failing = sum(1 for p in pass_fail if not p)
    for pred_key, outcomes in coverage_matrix.items():
        if not any(o == "unify_failed" for o in outcomes):
            continue
        ef = sum(1 for i, o in enumerate(outcomes) if o is not None and not pass_fail[i])
        ep = sum(1 for i, o in enumerate(outcomes) if o is not None and pass_fail[i])
        nf = n_failing - ef
        denom = math.sqrt((ef + nf) * (ef + ep))
        scores[pred_key] = ef / denom if denom > 0 else 0.0
    return scores


def compute_missing_predicate_scores(
    all_failed_subgoals: list[str], kb_pred_keys: set[str]
) -> dict[str, int]:
    counts: Counter = Counter()
    for subgoal in all_failed_subgoals:
        m = re.match(r"([a-z][a-zA-Z0-9_]*).*", subgoal)
        if m:
            pred_name = m.group(1)
            in_kb = any(k.split("/")[0] == pred_name for k in kb_pred_keys)
            if not in_kb:
                counts[subgoal] += 1
    return dict(counts)


def build_repair_agenda(
    ochiai_scores: dict[str, float],
    missing_scores: dict[str, int],
    k: int = K_REPAIR_TARGETS,
) -> list[dict[str, Any]]:
    max_o = max(ochiai_scores.values(), default=1.0) or 1.0
    max_m = max(missing_scores.values(), default=1) or 1
    agenda: list[dict] = []
    for pred, score in ochiai_scores.items():
        agenda.append({"item": pred, "score": score / max_o, "type": "erroneous"})
    for subgoal, freq in missing_scores.items():
        agenda.append({"item": subgoal, "score": freq / max_m, "type": "missing"})
    agenda.sort(key=lambda x: x["score"], reverse=True)
    return agenda[:k]


# Example: demonstrate Ochiai scoring on synthetic coverage data
coverage_matrix_ex = {
    "is_happy/1":  ["unify_failed", "unify_failed", None, "succeeded"],
    "parent_of/2": ["succeeded",    None,            "succeeded", "succeeded"],
}
pass_fail_ex = [False, False, True, True]  # first 2 queries fail, last 2 pass
scores_ex = compute_ochiai_scores(coverage_matrix_ex, pass_fail_ex)
print("Ochiai scores (higher = more suspicious):")
for pred, score in sorted(scores_ex.items(), key=lambda x: -x[1]):
    print(f"  {pred}: {score:.4f}")

## LLM Prompts

The pipeline uses 6 structured prompts:
1. **EXTRACTION_PROMPT** — NL premises → Prolog KB
2. **ORACLE_PROMPT** — Generate yes/no test questions about the premises
3. **ANSWER_PROMPT** — Ground-truth oracle answers to those questions
4. **BATCH_NL_TO_PROLOG_PROMPT** — Batch-convert oracle questions to Prolog goals (1 LLM call vs N)
5. **REPAIR_PROMPT** — Targeted clause repair for a suspicious predicate
6. **CONCLUSION_EVAL_PROMPT** — FOL conclusion → Prolog query for final evaluation

In [ ]:
EXTRACTION_PROMPT = """\
You are a Prolog knowledge base extractor. Given natural language premises, \
extract a Prolog knowledge base (facts and rules).

Rules:
- Use snake_case predicate names (e.g., is_happy/1, parent_of/2)
- Each fact/rule on its own line ending with '.'
- No comments, no module declarations, no :- use_module lines
- Use only alphanumeric and underscore in predicate/atom names
- Atoms that are proper nouns must be lowercase (e.g., john, bonnie)
- Output ONLY the Prolog code, nothing else

Premises:
{premises}

Prolog KB:"""

ORACLE_PROMPT = """\
Given these premises, generate {n} yes/no reasoning questions that test whether \
specific facts from the premises hold. Generate a mix: roughly half answerable 'yes', \
half 'no' or 'unknown'. Format: one question per line, starting with 'Q: '.

Premises:
{premises}

Questions:"""

ANSWER_PROMPT = """\
Based on the premises, answer each question with 'yes', 'no', or 'unknown'. \
Output ONLY 'yes', 'no', or 'unknown' for each question, one answer per line.

Premises:
{premises}

Questions:
{questions}

Answers (one per line):"""

BATCH_NL_TO_PROLOG_PROMPT = """\
Convert each yes/no question to a Prolog goal (without '?-').
Use ONLY predicates listed. Output one goal per line (no numbering, no explanation).
If a question cannot be converted, output: fail

Available KB predicates: {predicates}

Questions:
{questions}

Prolog goals (one per line):"""

REPAIR_PROMPT = """\
Repair this Prolog KB extracted from natural language. The predicate/sub-goal below \
is suspected erroneous or missing.

Premises:
{premises}

Current Prolog KB:
{kb}

Suspicious item: {item}
Reason: {reason}

Provide corrected/new Prolog clause(s). Output ONLY valid Prolog clauses, one per line, \
ending with '.'. No explanations."""

CONCLUSION_EVAL_PROMPT = """\
Convert this FOL conclusion to a Prolog goal (without '?-').
Use ONLY predicates listed below. Output exactly one line: the Prolog goal expression.
No explanations, no punctuation except the goal itself.
If it cannot be converted, output exactly: fail

Conclusion: {conclusion_fol}

Available KB predicates: {predicates}

Prolog goal (single line only):"""

print("All 6 LLM prompts defined.")

## Pre-Computed Results Analysis

Instead of running the full pipeline (which requires SWI-Prolog + OpenRouter API key), we analyze the pre-computed results from `mini_demo_data.json`. Each example contains predictions from all 5 methods and metadata from the SBFL repair process.

In [ ]:
examples = data["datasets"][0]["examples"]
method_names = ["dual_sbfl", "oneshot", "cot", "selfrefine", "flat_sbfl"]

print(f"Number of examples: {len(examples)}\n")
for i, ex in enumerate(examples):
    print(f"Example {i+1}:")
    print(f"  Gold label:     {ex['output']}")
    print(f"  Premises (truncated): {ex['input'][:100]}...")
    print(f"  Conclusion:     {ex['metadata_conclusion'][:80]}...")
    print(f"  KB size:        {ex['metadata_kb_size']} clauses")
    print(f"  Repairs made:   {ex['metadata_num_repairs']}")
    print(f"  Predictions:    " + ", ".join(f"{m}={ex[f'predict_{m}']}" for m in method_names))
    print(f"  Ochiai top-5:   {ex['metadata_ochiai_top5']}")
    print(f"  Subgoal top-5:  {ex['metadata_subgoal_top5'][:80]}...")
    print()

## Visualization: Method Accuracy Comparison

Compute per-method accuracy and display as a bar chart. Also show the summary statistics from the pre-computed run metadata.

In [ ]:
summary = data["metadata"]["summary"]
params = data["metadata"]["parameters"]

# Compute per-method accuracy from examples
correct = {m: 0 for m in method_names}
for ex in examples:
    gold = ex["output"]
    for m in method_names:
        if ex[f"predict_{m}"] == gold:
            correct[m] += 1

n = len(examples)
accuracies = {m: correct[m] / n for m in method_names}

# --- Print summary table ---
print("=" * 55)
print(f"{'Method':<20} {'Accuracy':>10}  {'Correct':>8}/{n}")
print("-" * 55)
for m in method_names:
    marker = " ← our method" if m == "dual_sbfl" else ""
    print(f"  {m:<18} {accuracies[m]:>10.1%}  {correct[m]:>8}{marker}")
print("=" * 55)
print(f"\nPipeline parameters: {params}")
print(f"Total LLM calls: {summary['total_llm_calls']}")
print(f"Total cost: ${summary['total_cost_usd']:.4f}")
print(f"Avg repairs/example: {summary['avg_repairs_per_example']}")

# --- Bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ["#2196F3" if m == "dual_sbfl" else "#90CAF9" for m in method_names]
bars = axes[0].bar(method_names, [accuracies[m] for m in method_names], color=colors)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Method Accuracy on FOLIO (mini demo, n=3)")
axes[0].set_xticklabels(method_names, rotation=20, ha="right")
for bar, m in zip(bars, method_names):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f"{accuracies[m]:.1%}", ha="center", va="bottom", fontsize=9)

# KB size and repairs per example
kb_sizes = [int(ex["metadata_kb_size"]) for ex in examples]
num_repairs = [int(ex["metadata_num_repairs"]) for ex in examples]
x = range(len(examples))
axes[1].bar([xi - 0.2 for xi in x], kb_sizes, width=0.35, label="KB clauses", color="#4CAF50")
axes[1].bar([xi + 0.2 for xi in x], num_repairs, width=0.35, label="Repairs made", color="#FF9800")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels([f"Ex {i+1}\n({ex['output']})" for i, ex in enumerate(examples)])
axes[1].set_ylabel("Count")
axes[1].set_title("KB Size and Repairs per Example")
axes[1].legend()

plt.tight_layout()
plt.savefig("results_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved results_summary.png")